# DGS Data, IO, and Loaders

This notebook is the entry point for the DGS usage-model tutorial series. It shows how raw genomic files become validated intervals, labels, sequence tensors, and PyTorch dataloaders.


## Tutorial series map

This notebook is part of the `Tutorials/0_DGS_usage_models` sequence:
- **`0_DGS_data.ipynb`: data, IO, intervals, targets, and loaders**
- `1_DGS_models.ipynb`: binary classification and regression scalar models
- `2_DGS_profile_models.ipynb`: profile/count and long-sequence track models
- `3_DGS_gLMs.ipynb`: genomic language model adapters and downstream heads
- `4_DGS_explain.ipynb`: interpretation methods, attribution artifacts, and motif discovery

The notebooks are designed to be read in order, but each one keeps its examples self-contained and guarded against missing optional dependencies.


## Audience, prerequisites, and learning goals

This notebook is for users who want to understand how DGS turns genomic files into tensors.

Prerequisites:

- Basic familiarity with FASTA and BED files.
- Basic PyTorch tensor concepts.
- Optional: BigWig knowledge for signal/profile tasks.

By the end, you should be able to:

- Explain the difference between `DGS.IO` readers and `DGS.Data` abstractions.
- Read FASTA and BED files with DGS readers.
- Create `Interval` and `Target` objects.
- Build sequence-only and supervised dataloaders.
- Understand when to use profile dataloaders for BigWig-backed profile models.
- Check tensor shape conventions before connecting data to a model.


## Outline

1. Set up a safe tutorial environment.
2. Create tiny FASTA and BED fixtures.
3. Read genomic files with `DGS.IO`.
4. Work with `Interval` and interval operations.
5. Convert sequences and inspect one-hot conventions.
6. Build sequence-only dataloaders.
7. Build supervised dataloaders with BED labels.
8. Use `Target` directly for target assembly and statistics.
9. Review BigWig and profile-loader templates.
10. Review pitfalls and exercises.


## Series-level conventions

Across these notebooks:

- Core examples use tiny synthetic data or local fixtures and avoid model downloads.
- Optional real-data or external-checkpoint cells are disabled with `RUN_*` flags until you edit paths and opt in.
- Loader sequence batches are usually `(batch, length, 4)`; CNN-style models usually expect `(batch, 4, length)` after `transpose(1, 2)`.
- Scalar targets use `(batch, tasks)`; profile targets use `(batch, tracks, profile_length)`.
- Environment guards print skip messages when optional packages such as `torch`, `pysam`, `pyBigWig`, `transformers`, or `evo2` are unavailable.


## 1. Environment setup

The core cells create tiny local files under `Tutorials/0_DGS_usage_models/_data_tutorial_tmp/`. They do not download external data.

Some IO operations require optional packages:

- `pysam` for indexed FASTA random access.
- `torch` for PyTorch dataloaders.
- `pyBigWig` for BigWig readers and profile loaders.

If a package is unavailable, the relevant cells print a skip message instead of failing.


In [ ]:
from pathlib import Path
import os
import sys

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "DGS").exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import numpy as np
    import pandas as pd
    if not hasattr(pd, "DataFrame"):
        raise ImportError("pandas imported without DataFrame; the environment may be inconsistent")
    PANDAS_READY = True
except Exception as exc:
    np = None
    pd = None
    PANDAS_READY = False
    PANDAS_IMPORT_ERROR = repr(exc)

try:
    import torch
    TORCH_AVAILABLE = True
except Exception as exc:
    torch = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = repr(exc)

try:
    import pysam
    PYSAM_AVAILABLE = True
except Exception as exc:
    pysam = None
    PYSAM_AVAILABLE = False
    PYSAM_IMPORT_ERROR = repr(exc)

try:
    import pyBigWig
    PYBIGWIG_AVAILABLE = True
except Exception as exc:
    pyBigWig = None
    PYBIGWIG_AVAILABLE = False
    PYBIGWIG_IMPORT_ERROR = repr(exc)

DGS_READY = False
if PANDAS_READY:
    try:
        from DGS.IO import FastaReader, BedReader, BigWigReader
        from DGS.IO.vcf import read_vcf
        from DGS.Data import (
            Interval,
            Target,
            find_overlaps,
            merge_intervals,
            get_reverse_complement,
            sequence_to_onehot,
            onehot_to_sequence,
            build_sequence_dataloader,
            build_supervised_dataloader,
            build_supervised_dataloaders,
            build_profile_dataloader,
        )
        DGS_READY = True
    except Exception as exc:
        DGS_IMPORT_ERROR = repr(exc)

print(f"Repository root: {REPO_ROOT}")
print(f"pandas/numpy ready: {PANDAS_READY}")
if not PANDAS_READY:
    print(f"pandas/numpy import error: {PANDAS_IMPORT_ERROR}")
print(f"torch available: {TORCH_AVAILABLE}")
if not TORCH_AVAILABLE:
    print(f"torch import error: {TORCH_IMPORT_ERROR}")
print(f"pysam available: {PYSAM_AVAILABLE}")
if not PYSAM_AVAILABLE:
    print(f"pysam import error: {PYSAM_IMPORT_ERROR}")
print(f"pyBigWig available: {PYBIGWIG_AVAILABLE}")
if not PYBIGWIG_AVAILABLE:
    print(f"pyBigWig import error: {PYBIGWIG_IMPORT_ERROR}")
print(f"DGS data API ready: {DGS_READY}")
if PANDAS_READY and not DGS_READY:
    print(f"DGS import error: {DGS_IMPORT_ERROR}")


## 2. Create tiny FASTA and BED fixtures

DGS follows standard genomics coordinate conventions:

- BED coordinates are zero-based and half-open: `start` is included, `end` is excluded.
- FASTA sequence extraction uses chromosome names that must match the BED file.
- For strand-aware loaders, negative-strand intervals are reverse-complemented.

The fixtures below are deliberately small so you can inspect every row.


In [ ]:
if DGS_READY:
    tutorial_dir = REPO_ROOT / "Tutorials" / "0_DGS_usage_models" / "_data_tutorial_tmp"
    tutorial_dir.mkdir(parents=True, exist_ok=True)

    fasta_path = tutorial_dir / "toy.fa"
    intervals_bed_path = tutorial_dir / "toy_intervals.bed"
    labels_bed_path = tutorial_dir / "toy_binary_labels.bed"
    vcf_path = tutorial_dir / "toy.vcf"

    chr1 = "ACGT" * 80
    chr2 = "TGCA" * 80
    fasta_path.write_text(f">chr1\n{chr1}\n>chr2\n{chr2}\n")

    intervals_df = pd.DataFrame(
        [
            ["chr1", 0, 24, "window_0", 0, "+"],
            ["chr1", 12, 36, "window_1", 0, "-"],
            ["chr1", 40, 64, "window_2", 0, "+"],
            ["chr2", 8, 32, "window_3", 0, "+"],
            ["chr2", 48, 72, "window_4", 0, "-"],
            ["chr2", 88, 112, "window_5", 0, "+"],
        ],
        columns=["chrom", "start", "end", "name", "score", "strand"],
    )
    intervals_df.to_csv(intervals_bed_path, sep="\t", header=False, index=False)

    labels_df = pd.DataFrame(
        [
            ["chr1", 5, 20, "1"],
            ["chr1", 45, 60, "1"],
            ["chr2", 90, 108, "1"],
        ],
        columns=["chrom", "start", "end", "name"],
    )
    labels_df.to_csv(labels_bed_path, sep="\t", header=False, index=False)

    vcf_path.write_text(
        "##fileformat=VCFv4.2\n"
        "#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n"
        "chr1\t10\tvar1\tC\tT\t.\tPASS\t.\n"
        "chr2\t51\tvar2\tG\tA\t.\tPASS\t.\n"
    )

    display(intervals_df)
    print("FASTA:", fasta_path)
    print("Intervals BED:", intervals_bed_path)
    print("Labels BED:", labels_bed_path)
    print("VCF:", vcf_path)
else:
    print("Skipped: DGS data API is not ready in this kernel.")


## 3. Read genomic files with `DGS.IO`

`DGS.IO` readers are low-level file readers. They validate file extensions, standardize columns, and expose file-specific operations.

Use them when you want to inspect files directly before building datasets.


In [ ]:
if DGS_READY and PYSAM_AVAILABLE:
    bed = BedReader(intervals_bed_path).read()
    display(bed)

    with FastaReader(fasta_path) as fasta:
        print("references:", fasta.references)
        print("lengths:", fasta.lengths)
        sequences = fasta.read(bed[["chrom", "start", "end"]])

    for name, sequence in zip(bed["name"], sequences):
        print(f"{name}: {sequence}")
elif DGS_READY:
    print("Skipped FASTA random access: pysam is not available.")
else:
    print("Skipped: DGS data API is not ready in this kernel.")


In [ ]:
if DGS_READY:
    variants = read_vcf(vcf_path)
    display(variants)
else:
    print("Skipped: DGS data API is not ready in this kernel.")


### Reader roles

- `BedReader` loads interval-like tables and standardizes columns such as `chrom`, `start`, and `end`.
- `FastaReader` extracts DNA strings from indexed FASTA files.
- `BigWigReader` reads continuous signal over intervals and can aggregate into bins.
- `read_vcf` parses core VCF columns for downstream variant workflows.

Most training workflows do not call these readers directly; they use higher-level dataloader builders.


## 4. Work with `Interval`

`Interval` wraps BED-like data and provides validation, summary statistics, overlap operations, and BED export.

Use `Interval` when you need to manipulate windows before creating datasets.


In [ ]:
if DGS_READY:
    intervals = Interval(intervals_bed_path)
    print("Interval stats:")
    display(intervals.get_stats())

    label_intervals = Interval(labels_bed_path)
    overlaps = intervals.merge_with(label_intervals, how="inner")
    print("Overlaps between query windows and label intervals:")
    display(overlaps.head())

    merged = intervals.merge_overlapping(min_distance=0)
    print("Merged intervals:")
    display(merged.data)
else:
    print("Skipped: DGS data API is not ready in this kernel.")


## 5. Convert sequences and inspect one-hot conventions

DGS sequence models usually consume one-hot DNA tensors.

Common shape conventions:

- Raw sequence strings: Python strings such as `"ACGT"`.
- Single one-hot sequence: often `(length, 4)`.
- Batched sequence-loader output: often `(batch, length, 4)`.
- Convolutional models such as `CNN`: often `(batch, 4, length)`, after `transpose(1, 2)`.


In [ ]:
if DGS_READY:
    toy_sequence = "ACGTNACGT"
    encoded = sequence_to_onehot(toy_sequence)
    decoded = onehot_to_sequence(encoded)

    print("sequence:", toy_sequence)
    print("one-hot shape:", encoded.shape)
    print("decoded:", decoded)
    print("reverse complement:", get_reverse_complement(toy_sequence))
else:
    print("Skipped: DGS data API is not ready in this kernel.")


## 6. Build a sequence-only dataloader

Use `build_sequence_dataloader` when you only need sequences, for example for embedding extraction, sequence-only inference, or debugging model input shapes.

The returned batches use channel-last one-hot encoding: `(batch, length, 4)`.


In [ ]:
if DGS_READY and TORCH_AVAILABLE and PYSAM_AVAILABLE:
    sequence_loader = build_sequence_dataloader(
        fasta_path=fasta_path,
        intervals_path=intervals_bed_path,
        batch_size=2,
        mode="streaming",
        strand_aware=True,
        shuffle=False,
    )

    sequence_batch = next(iter(sequence_loader))
    print("sequence batch shape:", tuple(sequence_batch.shape), "= (batch, length, 4)")
    print("channel-first for CNN:", tuple(sequence_batch.transpose(1, 2).contiguous().shape), "= (batch, 4, length)")
elif DGS_READY:
    print("Skipped sequence dataloader: torch and pysam are required.")
else:
    print("Skipped: DGS data API is not ready in this kernel.")


## 7. Build a supervised dataloader with BED labels

Use `build_supervised_dataloader` when each interval should receive scalar target labels.

Target task definitions require:

- `task_name`: a stable output name.
- `file_path`: a BED or BigWig target file.
- `file_type`: `"bed"` or `"bigwig"`.
- `task_type`: commonly `"binary"` or `"regression"`.

For BED targets, overlapping label intervals assign positive labels to query windows.


In [ ]:
if DGS_READY and TORCH_AVAILABLE and PYSAM_AVAILABLE:
    target_tasks = [
        {
            "task_name": "toy_overlap_label",
            "file_path": str(labels_bed_path),
            "file_type": "bed",
            "task_type": "binary",
            "target_column": "name",
        }
    ]

    supervised_loader = build_supervised_dataloader(
        fasta_path=fasta_path,
        intervals_path=intervals_bed_path,
        target_tasks=target_tasks,
        batch_size=3,
        mode="streaming",
        strand_aware=True,
        shuffle=False,
    )

    sequences, targets = next(iter(supervised_loader))
    print("sequences:", tuple(sequences.shape), "= (batch, length, 4)")
    print("targets:", tuple(targets.shape), "= (batch, tasks)")
    print("targets values:", targets.squeeze(-1).tolist())
elif DGS_READY:
    print("Skipped supervised dataloader: torch and pysam are required.")
else:
    print("Skipped: DGS data API is not ready in this kernel.")


In [ ]:
if DGS_READY and TORCH_AVAILABLE and PYSAM_AVAILABLE:
    train_loader, val_loader, test_loader = build_supervised_dataloaders(
        fasta_path=fasta_path,
        intervals_path=intervals_bed_path,
        target_tasks=target_tasks,
        batch_size=2,
        mode="streaming",
        split="random",
        test_size=0.25,
        val_size=0.25,
        random_state=7,
        strand_aware=True,
    )

    print("train batches:", len(train_loader))
    print("val batches:", len(val_loader))
    print("test batches:", len(test_loader))
else:
    print("Skipped train/val/test loader example: torch and pysam are required.")


## 8. Use `Target` directly

`Target` assembles labels or signal values aligned to a set of query intervals. Dataloaders use it internally, but direct use is helpful for debugging target definitions before training.


In [ ]:
if DGS_READY:
    target = Target(intervals_df, target_tasks)
    print("available tasks:", list(target.tasks.keys()))
    print("task info:")
    display(target.get_task_info())
    print("raw target array:")
    print(target.data["toy_overlap_label"])
else:
    print("Skipped: DGS data API is not ready in this kernel.")


## 9. BigWig and profile-loader templates

BigWig-backed workflows appear in two forms:

- Scalar supervised targets: aggregate signal over each interval, often for regression.
- Profile targets: keep a fixed-width signal vector for each interval, used by BPNet/ChromBPNet-style profile models.

The following cells are templates and are disabled by default because they require `pyBigWig` and real signal files.


In [ ]:
RUN_BIGWIG_SCALAR_TEMPLATE = False

if RUN_BIGWIG_SCALAR_TEMPLATE:
    bigwig_tasks = [
        {
            "task_name": "mean_signal",
            "file_path": "path/to/signal.bw",
            "file_type": "bigwig",
            "task_type": "regression",
            "bin_size": None,
            "aggfunc": "mean",
        }
    ]

    loader = build_supervised_dataloader(
        fasta_path="path/to/genome.fa",
        intervals_path="path/to/windows.bed",
        target_tasks=bigwig_tasks,
        batch_size=32,
        mode="streaming",
    )
    sequences, targets = next(iter(loader))
    print(sequences.shape, targets.shape)
else:
    print("BigWig scalar target template is disabled. Set RUN_BIGWIG_SCALAR_TEMPLATE = True after editing paths.")


In [ ]:
RUN_PROFILE_LOADER_TEMPLATE = False

if RUN_PROFILE_LOADER_TEMPLATE:
    profile_tasks = [
        {
            "task_name": "plus_signal_profile",
            "file_path": "path/to/plus_signal.bw",
            "file_type": "bigwig",
            "task_type": "profile",
            "bin_size": 1,
            "aggfunc": None,
        }
    ]

    profile_loader = build_profile_dataloader(
        fasta_path="path/to/genome.fa",
        intervals_path="path/to/fixed_width_windows.bed",
        profile_tasks=profile_tasks,
        batch_size=8,
        strand_aware=True,
        return_counts=True,
    )

    sequences, (profiles, counts) = next(iter(profile_loader))
    print("sequences:", sequences.shape)  # (batch, length, 4)
    print("profiles:", profiles.shape)    # (batch, tasks, profile_length)
    print("counts:", counts.shape)        # (batch, tasks)
else:
    print("Profile loader template is disabled. Set RUN_PROFILE_LOADER_TEMPLATE = True after editing paths.")


## 10. How to choose the right loader

| Goal | Recommended API | Batch output |
|---|---|---|
| Sequence-only inference or embeddings | `build_sequence_dataloader` | `sequences` |
| Binary or regression scalar targets | `build_supervised_dataloader` | `(sequences, targets)` |
| Train/val/test scalar workflow | `build_supervised_dataloaders` | three loaders |
| Profile/count targets from BigWig | `build_profile_dataloader` | `(sequences, profiles)` or `(sequences, (profiles, counts))` |
| Direct file inspection | `FastaReader`, `BedReader`, `BigWigReader` | file-specific objects/arrays |
| Interval manipulation | `Interval` | validated interval DataFrame |
| Target debugging | `Target` | task-aligned arrays and stats |

For DGS convolutional models, remember to transpose sequence batches from `(batch, length, 4)` to `(batch, 4, length)` before model input.


## 11. Exercise

Modify the supervised dataloader example to add a second binary task.

Suggested steps:

- Create another BED file with different positive intervals.
- Add a second dictionary to `target_tasks`.
- Rebuild `build_supervised_dataloader`.
- Confirm that target batches have shape `(batch, 2)`.


In [ ]:
if DGS_READY and TORCH_AVAILABLE and PYSAM_AVAILABLE:
    second_labels_path = tutorial_dir / "toy_second_labels.bed"
    second_labels_df = pd.DataFrame(
        [
            ["chr1", 0, 10, "1"],
            ["chr2", 45, 70, "1"],
        ],
        columns=["chrom", "start", "end", "name"],
    )
    second_labels_df.to_csv(second_labels_path, sep="\t", header=False, index=False)

    two_task_target_tasks = target_tasks + [
        {
            "task_name": "second_label",
            "file_path": str(second_labels_path),
            "file_type": "bed",
            "task_type": "binary",
            "target_column": "name",
        }
    ]

    two_task_loader = build_supervised_dataloader(
        fasta_path=fasta_path,
        intervals_path=intervals_bed_path,
        target_tasks=two_task_target_tasks,
        batch_size=4,
        mode="streaming",
        strand_aware=True,
        shuffle=False,
    )
    _, two_task_targets = next(iter(two_task_loader))
    print("two-task targets shape:", tuple(two_task_targets.shape))
    print(two_task_targets)
else:
    print("Skipped exercise execution: torch, pysam, and DGS data API are required.")


## Common pitfalls

Pitfall 1: chromosome names do not match.

- `chr1` and `1` are different names.
- FASTA, BED, BigWig, and VCF files must use the same naming convention.

Pitfall 2: coordinate conventions are mixed.

- BED is zero-based half-open.
- VCF `POS` is one-based.
- Convert carefully when moving between formats.

Pitfall 3: interval widths are inconsistent for profile models.

- Profile loaders require fixed-width intervals because profile tensors must stack into a rectangular batch.

Pitfall 4: sequence orientation is wrong.

- Loader batches are generally `(batch, length, 4)`.
- CNN-like models generally require `(batch, 4, length)`.

Pitfall 5: target task keys are inconsistent.

- High-level loaders require `task_name`, `file_path`, and `file_type`.
- Profile loaders currently require BigWig files.


## Optional extensions

- Replace the tiny fixtures with one chromosome from a real genome build.
- Use chromosome-based splits with held-out chromosomes.
- Add BigWig scalar regression targets.
- Add profile/count targets and connect them to the profile-model tutorial.
- Add VCF-centered sequence extraction for variant-effect prediction workflows.
